# MoodLens — Warm-start fine-tune (v3)

v1 and v2 both fine-tuned from *raw* BERT and both lost to the baseline (0.663 macro-F1).
This round pulls the one lever we never tried: **warm start**.

- **Warm start** — begin from the already emotion-tuned encoder of
  `bhadresh-savani/bert-base-go-emotion` (the baseline's own backbone), keep its learned
  representations, and bolt on a fresh 7-class Ekman head.
- **Single-label training** — standard cross-entropy on the single-label rows, so the training
  objective matches the single-label argmax **evaluation** exactly (v2's multi-label/argmax
  mismatch is gone).
- **Weighting: off vs mild** — v2 showed heavy class weighting *hurt* macro-F1. We test
  unweighted vs mild-weighted (capped) and let the numbers decide.

Scored on the **same 4,968-row single-label test set** as the baseline → macro-F1 compares
directly to **0.663**.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
~30–50 min for both variants. Downloads the best model as a zip.

## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q transformers==4.47.1 datasets==3.2.0 scikit-learn==1.6.0 accelerate==1.2.1

## 3. Ekman labels + GoEmotions mapping (mirrors app/core/emotions.py)

In [ ]:
EKMAN = ["joy", "anger", "sadness", "fear", "surprise", "disgust", "neutral"]
EKMAN_LABEL2ID = {e: i for i, e in enumerate(EKMAN)}
EKMAN_ID2LABEL = {i: e for e, i in EKMAN_LABEL2ID.items()}

GOEMOTIONS_TO_EKMAN = {
    "amusement": "joy", "excitement": "joy", "joy": "joy", "love": "joy",
    "desire": "joy", "optimism": "joy", "caring": "joy", "pride": "joy",
    "admiration": "joy", "gratitude": "joy", "relief": "joy", "approval": "joy",
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    "sadness": "sadness", "disappointment": "sadness", "embarrassment": "sadness",
    "grief": "sadness", "remorse": "sadness",
    "fear": "fear", "nervousness": "fear",
    "surprise": "surprise", "realization": "surprise", "confusion": "surprise",
    "curiosity": "surprise",
    "disgust": "disgust",
    "neutral": "neutral",
}

WARM_BASE = "bhadresh-savani/bert-base-go-emotion"

## 4. Build single-label datasets
Keep only rows that map to exactly one Ekman bucket — the same protocol the baseline was
scored on. Integer labels (single-label), so training uses plain cross-entropy.

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

raw = load_dataset("go_emotions", "simplified")
go_names = raw["train"].features["labels"].feature.names

def convert(split):
    texts, labels = [], []
    for ex in split:
        buckets = {GOEMOTIONS_TO_EKMAN.get(go_names[i]) for i in ex["labels"]}
        buckets.discard(None)
        if len(buckets) != 1:
            continue
        emo = next(iter(buckets))
        texts.append(ex["text"]); labels.append(EKMAN_LABEL2ID[emo])
    return Dataset.from_dict({"text": texts, "label": labels})

ds = DatasetDict({s: convert(raw[s]) for s in raw})
for s, d in ds.items():
    counts = {e: 0 for e in EKMAN}
    for l in d["label"]:
        counts[EKMAN[l]] += 1
    print(f"[{s}] {len(d)} rows -> {counts}")

## 5. Mild class weights (for the weighted variant only)
`sqrt(N / count)` normalised, capped at 3 — a *gentle* nudge for rare classes, not the ×10
sledgehammer that hurt v2. The unweighted variant ignores these.

In [ ]:
import torch, numpy as np

train_counts = torch.zeros(len(EKMAN))
for l in ds["train"]["label"]:
    train_counts[l] += 1
N = int(train_counts.sum())
class_weights = (N / train_counts).sqrt()
class_weights = (class_weights / class_weights.mean()).clamp(max=3.0)
for e, c, w in zip(EKMAN, train_counts.tolist(), class_weights.tolist()):
    print(f"{e:9s} n={int(c):5d}  weight={w:.2f}")

## 6. Training helper
Warm-starts from the go-emotion encoder (`ignore_mismatched_sizes=True` drops the old 28-label
head and inits a fresh 7-Ekman head, keeping the encoder). Optional class-weighted
cross-entropy. Metrics = single-label argmax accuracy + macro-F1 — identical to the baseline.

In [ ]:
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback)


class WeightedCETrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro", zero_division=0)}


def train_and_eval(tag, weights=None, epochs=5, batch_size=32, lr=2e-5):
    tok = AutoTokenizer.from_pretrained(WARM_BASE)
    ds_tok = ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        WARM_BASE, num_labels=len(EKMAN), ignore_mismatched_sizes=True,
        id2label=EKMAN_ID2LABEL, label2id=EKMAN_LABEL2ID)

    args = TrainingArguments(
        output_dir=f"ckpt-{tag}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, save_total_limit=1, logging_steps=100, report_to="none")

    trainer = WeightedCETrainer(
        class_weights=weights, model=model, args=args,
        train_dataset=ds_tok["train"], eval_dataset=ds_tok["validation"],
        tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

    trainer.train()
    res = trainer.evaluate(ds_tok["test"])
    trainer.save_model(f"ekman-{tag}"); tok.save_pretrained(f"ekman-{tag}")
    print(f"\n=== {tag} TEST ===  acc={res['eval_accuracy']:.4f}  macro_f1={res['eval_macro_f1']:.4f}\n")
    return {"accuracy": round(res["eval_accuracy"], 4), "macro_f1": round(res["eval_macro_f1"], 4)}

## 7. Train both variants (warm-start, unweighted vs mild-weighted)

In [ ]:
RESULTS = {"baseline (off-the-shelf)": {"accuracy": 0.722, "macro_f1": 0.663}}
print("\n########## warm-start, UNWEIGHTED ##########\n")
RESULTS["warm-unweighted"] = train_and_eval("warm-unweighted", weights=None)
print("\n########## warm-start, MILD-WEIGHTED ##########\n")
RESULTS["warm-weighted"] = train_and_eval("warm-weighted", weights=class_weights)

## 8. Comparison table

In [ ]:
print(f"{'model':28s} {'accuracy':>10s} {'macro_f1':>10s}")
print("-" * 50)
for name, m in RESULTS.items():
    print(f"{name:28s} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f}")

trained = {k: v for k, v in RESULTS.items() if k != "baseline (off-the-shelf)"}
best = max(trained, key=lambda k: trained[k]["macro_f1"])
beat = trained[best]["macro_f1"] > RESULTS["baseline (off-the-shelf)"]["macro_f1"]
print(f"\nBest: {best} (macro_f1={trained[best]['macro_f1']}) — "
      f"{'BEATS' if beat else 'does NOT beat'} baseline.")

## 9. Download the best model
Unzip into `backend/models/ekman-best/`, set `MODEL_NAME=models/ekman-best` in `backend/.env`.

Note: this model has a native 7-Ekman head, so serving it needs the classifier to skip the
28→7 aggregation. Flag it when you wire it in.

In [ ]:
import shutil
shutil.make_archive(f"ekman-{best}", "zip", f"ekman-{best}")
from google.colab import files
files.download(f"ekman-{best}.zip")